# MCP

In [1]:
import os

import chromadb
import dotenv
from agents import Agent, Runner, function_tool, trace
from agents.mcp import MCPServerStreamableHttp

dotenv.load_dotenv()

True

Let's set up our RAG database connection:

In [2]:
chroma_client = chromadb.PersistentClient(path="../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [3]:
# This is the same code as in the rag.ipynb notebook


@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Integrate EXA Search as an MCP:  NOTE: ALWAYS PUT GUARDRAILS ON YOUR AGENTS WITH instructions=  docstring

In [ ]:
# Increased EXA timeout from 30 to 90 seconds since EXA can take longer than 30 seconds to respond when under heavy load
############################### Here is how we connect to our MCP server for Exa Web Search ###########
exa_search_mcp = MCPServerStreamableHttp(
    name="Exa Search MCP",
    params={
        "url": f"https://mcp.exa.ai/mcp?exaApiKey={os.environ.get("EXA_API_KEY")}",
        "timeout": 90,
    },
    client_session_timeout_seconds=90,
    cache_tools_list=True, # get the list of tools available at this MCP server only once, caches it
    max_retry_attempts=1,
)
##################################### Next, Connect to your MCP server below ##########################

await exa_search_mcp.connect()

#########################################################################################
######## Use the instructions= docstring to ALWAYS constraint your agent with guardrails
##########################################################################################
calorie_agent_with_search = Agent(
    name="Nutrition Assistant",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, search the EXA web to figure out the exact ingredients of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, if necessary, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use Exa Search to find the exact recipe and ingredients.
    * Once you know the ingredients, use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 10 times.
    """,
    tools=[calorie_lookup_tool],
    mcp_servers=[exa_search_mcp],   ### beside tools, now we have a list of available MCP servers(and their tools)
)

Reference query - shouldn't use ExaSearch:(because the answer is in our RAG Vector Store already)
You can verify this by checking its' trace at https://platform.openai.com/logs?api=traces

In [5]:
with trace("Nutrition Assistant with MCP - Only uses calorie_lookup_tool"):
    result = await Runner.run(
        calorie_agent_with_search,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    - Banana (1 medium, ~118 g): ~105 kcal
    - Apple (1 medium, ~182 g): ~95 kcal
    - Total: ~200 kcal
    
    Calories per 100 g:
    - Banana: 89 kcal/100 g
    - Apple: 52 kcal/100 g
- 7 new item(s)
- 2 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


#### The next cell can not use the RAG Vector Store because english breakfast is a meal not a fruit, so it must our Exa MCP server to do a Web Search only and actually the trace will show that the calorie_lookup_tool was never used. You can verify the trace for yourself at https://platform.openai.com/logs?api=traces

In [6]:
with trace("Nutrition Assistant with MCP "):
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an english breakfast?"
    )
    print(result.final_output)

A traditional full English breakfast is typically about 800–900 calories per serving, depending on portions and extras.

Example breakdown (rough estimates):
- 2 eggs: ~140 kcal
- 2 slices bacon: ~90 kcal
- 1 pork sausage: ~230 kcal
- 1/2 cup baked beans: ~150 kcal
- 1 grilled tomato: ~15 kcal
- Sautéed mushrooms: ~20 kcal
- 2 slices toast with butter: ~150 kcal

Total: ~800–900 kcal. If portions differ, calories vary accordingly.


### Assignment: replace Exa MCP with OpenAI hosted tools WebSearchTool() 

In [8]:
from agents import WebSearchTool
calorie_agent_with_search = Agent(
    name="Nutrition Assistant",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, search the web to figure out the exact ingredients of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, if it's about a meal, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use web search to find the exact recipe and ingredients.
    * Once you know the ingredients, always use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 8 times.
    """,
    tools=[calorie_lookup_tool, WebSearchTool()],
)

In [9]:
with trace("Nutrition Assistant with OpenAI Hosted Tools Web Search"):
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an english breakfast?"
    )
    print(result.final_output)

A typical full English breakfast is about 700–900 kcal per serving, potentially up to ~1000 kcal depending on portions (eggs, bacon, sausage, beans, tomatoes, mushrooms, and toasted bread). If you share exact portions or brands, I can calculate a precise total.
